# Patronage_Flow Pipeline v2 — Singapore footpath pedestrian flow

**Stack**: madina (modern UNA) on top of OSM Highway + LTA ridership + URA buildings.

**Algorithm — patronage betweenness** (Sevtsuk / madina):
For each origin (transit station, weight = ridership) and each destination 
(building, weight = floor area) within the walking catchment, compute the shortest path. 
Distribute the trip volume across edges with exponential distance decay and Huff-style 
destination competition. Sum across all O-D pairs to get per-edge pedestrian flow.

v2 differs from v1 (NKDE radial decay) in three ways:
- network is the OSM `highway` layer filtered to pedestrian-passable tags (replaces the 
non-routable LTA Footpath layer)
- destinations are buildings weighted by floor area (replaces the destination-less radial decay)
- engine is **madina** with proper noding, snap, turn penalty, parallelism

**Outputs**: per-edge `flow` value for each (DAY_TYPE × HOUR) bin you select.


## 0. Setup
Working directory: `D:/Claude/UNA/`. The `madina` package is installed in editable mode 
from `madina-main/`.

In [ ]:
import sys, importlib, time, warnings
from pathlib import Path
warnings.simplefilter('ignore')

ROOT = Path(r'D:/Claude/UNA')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

from Patronage_Flow import Constants as C
from Patronage_Flow import Network, Flow_Computation, Main
for m in (C, Network, Flow_Computation, Main):
    importlib.reload(m)

import madina
print('madina version :', madina.__version__ if hasattr(madina, "__version__") else 'unknown')
print('Patronage_Flow loaded. ROOT =', ROOT)


## 1. Verify inputs

In [ ]:
inputs = {
    'highway gpkg':   C.HIGHWAY_GPKG,
    'bus stop shp':   C.BUS_SHP,
    'MRT polygon shp':C.MRT_SHP,
    'bus ridership':  C.BUS_CSV,
    'train ridership':C.TRAIN_CSV,
    'MRT lookup':     C.MRT_LOOKUP_CSV,
    'building shp':   C.BUILDING_SHP,
}
for label, p in inputs.items():
    flag = 'OK ' if p.exists() else 'MISSING'
    sz   = f'{p.stat().st_size/1e6:7.1f} MB' if p.exists() else ''
    print(f'{flag} {label:<16} {sz}  {p.name}')

print('\npedestrian highway tags included:')
print(' ', ' '.join(C.PEDESTRIAN_HIGHWAYS))


## 2. Run mode
Smoke bbox first to verify the pipeline. Then `SMOKE = False` for the full island.

In [ ]:
SMOKE = True
if SMOKE:
    C.SMOKE_BBOX = (27000, 28000, 33000, 34500)   # Bras Basah / Marina
    C.HOURS_WEEKDAY = [8, 13, 18]
    C.HOURS_WEEKEND = [13]
    print('SMOKE TEST mode  --  bbox =', C.SMOKE_BBOX,
          ' hours WD/WE =', C.HOURS_WEEKDAY, C.HOURS_WEEKEND)
else:
    C.SMOKE_BBOX = None
    C.HOURS_WEEKDAY = list(range(24))
    C.HOURS_WEEKEND = list(range(24))
    print('FULL ISLAND mode  --  expect ~hours of compute')


## 3. Load and filter the OSM highway network
Only `highway` tags in `C.PEDESTRIAN_HIGHWAYS` survive. We additionally drop rows 
with `foot=no/private` or `access=no/private`.

In [ ]:
bbox = C.SMOKE_BBOX
hw = Network.load_pedestrian_highway(bbox=bbox)
print('\nfinal pedestrian highway features:', len(hw))

if SMOKE:
    fig, ax = plt.subplots(figsize=(9, 6))
    hw.plot(ax=ax, color='#666', linewidth=0.4)
    ax.set_title(f'Pedestrian highway -- {len(hw):,} segments')
    ax.set_aspect('equal'); ax.set_axis_off()
    plt.show()


## 4. Load stations and buildings

In [ ]:
stations = Network.load_stations(bbox=bbox).reset_index(drop=True)
buildings = Network.load_buildings(bbox=bbox).reset_index(drop=True)
print()
print('stations preview:'); print(stations.head(5).drop(columns='geometry'))
print()
print('buildings preview:'); print(buildings.head(5).drop(columns='geometry'))


## 5. Build the madina Zonal
1. `create_street_network` -- topology, snapping, redundant-edge handling
2. `insert_node('stations', label='origin', weight_attribute='W_init')` -- placeholder weight 1
3. `insert_node('buildings', label='destination', weight_attribute='W')` -- floor area
4. `create_graph()` -- internal NetworkX graphs ready for path-finding


In [ ]:
z = Network.build_zonal(hw, stations, buildings)
print('\nedges :', len(z.network.edges))
print('nodes :', len(z.network.nodes), 'breakdown =',
      z.network.nodes.type.value_counts().to_dict())


### Visualize the inserted origins on the network (smoke)

In [ ]:
if SMOKE:
    fig, ax = plt.subplots(figsize=(9, 6))
    z.network.edges.plot(ax=ax, color='#bbb', linewidth=0.3)
    nd = z.network.nodes
    nd[nd.type == 'origin'].plot(ax=ax, color='#3a7', markersize=8, label='origin (station)')
    nd[nd.type == 'destination'].plot(ax=ax, color='#c33', markersize=2, alpha=0.4, label='destination (building)')
    ax.legend(); ax.set_aspect('equal'); ax.set_axis_off()
    plt.show()


## 6. Hourly betweenness flow
Loop over each (DAY_TYPE, HOUR), set per-station ridership weights on the origin nodes, 
and call `madina.una.tools.betweenness()`. Each call writes a column 
`flow_<daytype>_<HH>` onto the edge GeoDataFrame. The per-call runtime scales roughly 
linearly with the number of origins (stations) inside the bbox.

In [ ]:
ridership = Flow_Computation.load_ridership_table()
print('sample ridership row (Orchard NS22):')
if 'NS22' in ridership.index:
    print(ridership.loc['NS22'].unstack(0).round(0).to_string())

flow_long = Flow_Computation.compute_hourly_flow(
    z, stations, ridership,
    hours_weekday=C.HOURS_WEEKDAY,
    hours_weekend=C.HOURS_WEEKEND,
    radius=C.RADIUS_M, beta=C.BETA, n_workers=C.N_WORKERS,
)
print('\nflow_long shape:', flow_long.shape)
print(flow_long.head(8).to_string())


## 7. QC: distribution and top edges
Most edges are out of any station's 800 m catchment, so median flow is 0. 
What matters is the upper tail.

In [ ]:
print('per-(daytype, hour) summary:')
g = flow_long.groupby(['DAY_TYPE','HOUR'])['flow']
print(g.agg(['mean', lambda s: s.quantile(0.9), lambda s: s.quantile(0.99), 'max'])
        .rename(columns={'<lambda_0>':'p90', '<lambda_1>':'p99'}).round(2))

print('\ntop 10 edges WD 18:00')
print(flow_long.query('DAY_TYPE == "WEEKDAY" and HOUR == 18')
        .nlargest(10, 'flow')[['edge_id','flow']].to_string(index=False))


## 8. Visualize hourly flow maps (smoke)
AM peak / midday / PM peak. Log scale because the distribution is heavy-tailed.

In [ ]:
if SMOKE:
    show = [(h, f'WEEKDAY {h:02d}:00') for h in C.HOURS_WEEKDAY[:3]]
    fig, axes = plt.subplots(1, len(show), figsize=(6*len(show), 6))
    if len(show) == 1: axes = [axes]
    for ax, (h, title) in zip(axes, show):
        sub = flow_long.query('DAY_TYPE == "WEEKDAY" and HOUR == @h')
        gdf = z.network.edges.copy()
        gdf['flow'] = sub.set_index('edge_id').reindex(gdf.index)['flow'].fillna(0).clip(lower=0.001)
        gdf.plot(ax=ax, column='flow', cmap='magma',
                 norm=LogNorm(vmin=1, vmax=max(100, gdf['flow'].max())),
                 linewidth=1.0)
        ax.set_title(title); ax.set_aspect('equal'); ax.set_axis_off()
    plt.tight_layout(); plt.show()


## 9. Export to GeoPackage for QGIS / Rhino

In [ ]:
C.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
z.network.edges.to_file(C.OUTPUT_DIR / 'edges.gpkg', driver='GPKG')
flow_long.to_parquet(C.OUTPUT_DIR / 'flow_long.parquet', index=False)

for (daytype, h), sub in flow_long.groupby(['DAY_TYPE','HOUR']):
    out = z.network.edges.copy()
    out['flow'] = sub.set_index('edge_id').reindex(out.index)['flow'].fillna(0.0)
    tag = f"{daytype.split('/')[0].lower()}_{h:02d}"
    out.to_file(C.OUTPUT_DIR / f'flow_{tag}.gpkg', driver='GPKG')
    print(f'wrote flow_{tag}.gpkg  p99={out.flow.quantile(0.99):8.1f}  max={out.flow.max():9.1f}')


## 10. **FULL ISLAND RUN**
Once the smoke results look right, switch off the bbox and bump the hour list. 
Single-core estimate: ~10–15 sec/origin × 5,400 origins ≈ 70 min per (daytype,hour). 
For 48 hours, plan a multi-hour or overnight run, or restrict to representative hours.

Tip: madina supports `num_cores > 1`; test with 2 cores first since Windows multiprocessing 
is finicky. Bump `C.N_WORKERS` and re-run.

In [ ]:
# --- FULL ISLAND ---
C.SMOKE_BBOX     = None
C.HOURS_WEEKDAY  = list(range(24))
C.HOURS_WEEKEND  = list(range(24))
# C.N_WORKERS = 2   # try 2 first; raise to 4-8 if stable

for m in (Network, Flow_Computation, Main):
    importlib.reload(m)

t0 = time.perf_counter()
Main.main()
print(f'\n=== full island run done in {(time.perf_counter()-t0)/60:.1f} min ===')


## 11. Pair flow with shade
The output `flow_<daytype>_<HH>.gpkg` files share their geometry with whatever shade 
raster you computed (buildings + trees + shelters). Standard workflow:
1. Buffer each edge by ~1.5 m to get a footpath polygon strip
2. Compute mean shade fraction inside that strip per hour
3. Join shade × flow on edge_id and DAY_TYPE × HOUR
4. Plot 2D scatter: shade fraction vs. flow density. The 'low shade × high flow' 
quadrant is the priority for new shelter / tree planting.